### Кодирование категориальных признаков

In [216]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

In [217]:
DATA_PREPROCESSED_PATH = "../data/processed/diabetic_data_processed.csv"

TOP_N_UNIQUE_VALUES = 5

In [218]:
df = pd.read_csv(DATA_PREPROCESSED_PATH)

df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,miglitol,tolazamide,insulin,glyburide-metformin,change,diabetesMed,target,glu_measured,a1c_measured,weight_measured
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Other,...,No,No,No,No,No,No,0,0,0,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Other,...,No,No,Up,No,Ch,Yes,0,0,0,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Other,...,No,No,No,No,No,Yes,0,0,0,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Other,...,No,No,Up,No,Ch,Yes,0,0,0,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Other,...,No,No,Steady,No,Ch,Yes,0,0,0,0


In [219]:
def top_values(series, n=10):
    value_counts = series.value_counts(dropna=False).head(n)
    return ", ".join(f"{val} ({count})" for val, count in value_counts.items())

In [220]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "nunique": df.nunique(dropna=False),
    "NaN count": df.isna().sum(),
    f"unique (top {TOP_N_UNIQUE_VALUES})": df.apply(lambda x: top_values(x, TOP_N_UNIQUE_VALUES)),
})

print(summary.to_string())

                          dtype  nunique  NaN count                                                                                                      unique (top 5)
encounter_id              int64   101766          0                                                           2278392 (1), 149190 (1), 64410 (1), 500364 (1), 16680 (1)
patient_nbr               int64    71518          0                                            88785891 (40), 43140906 (28), 1660293 (23), 23199021 (23), 88227540 (23)
race                        str        6          0                           Caucasian (76099), AfricanAmerican (19210), Unknown (2273), Hispanic (2037), Other (1506)
gender                      str        3          0                                                                   Female (54708), Male (47055), Unknown/Invalid (3)
age                         str       10          0                                  [70-80) (26068), [60-70) (22483), [50-60) (17256), [80-90) (17197), [40-50)

Некоторые категории в признаках слишком малы, эти строки можно удалить без потери важных данных.

Например, в `gender` строки с категорией `Unknown/Invalid` можно удалить, их слишком мало

При этом, аналогичные строки в признаках мед. препаратов не стоит удалять, поскольку они могут нести важную информацию

In [221]:
print(f"Строк до удаления: {len(df[df['gender'] == 'Unknown/Invalid'])}")

df = df[df["gender"] != "Unknown/Invalid"].reset_index(drop=True)

print(f"Строк после удаления: {len(df[df['gender'] == 'Unknown/Invalid'])}")

Строк до удаления: 3
Строк после удаления: 0


Определим способ кодирования для каждого категориального признака:

`gender` -  бинарный признак, `label-encoding`

`age`, все признаки медицинских препаратов - порядковые категориальные признаки, подойдет `ordinal-encoder`

`race`, `diag_1`, `diag_2`, `diag_3`, `medical_specialty` и `payer_code` - номинальные категориальные признаки, подойдет `one-hot-encoding`

In [222]:
df_encoded = df.copy()

#### Label Encoding

In [223]:
print("Столбец gender до кодирования")

df_encoded["gender"].head()

Столбец gender до кодирования


0    Female
1    Female
2    Female
3      Male
4      Male
Name: gender, dtype: str

In [224]:
label_enc = LabelEncoder()
df_encoded["gender"] = label_enc.fit_transform(df_encoded["gender"])

print("Столбец gender после кодирования")

df_encoded["gender"].head()

Столбец gender после кодирования


0    0
1    0
2    0
3    1
4    1
Name: gender, dtype: int64

#### Ordinal Encoding

In [225]:
ordinal_cols = [
    "age", "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "glipizide", "glyburide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "tolazamide", "insulin", "glyburide-metformin",
]

age_order = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]
medical_order = ["No", "Steady", "Down", "Up"]
tolazamide_order = ['No', 'Steady', 'Up']

In [226]:
print("Порядковые столбцы до кодирования")

df_encoded[ordinal_cols].head()

Порядковые столбцы до кодирования


,age,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,miglitol,tolazamide,insulin,glyburide-metformin
0,[0-10),No,No,No,No,No,No,No,No,No,No,No,No,No,No
1,[10-20),No,No,No,No,No,No,No,No,No,No,No,No,Up,No
2,[20-30),No,No,No,No,No,Steady,No,No,No,No,No,No,No,No
3,[30-40),No,No,No,No,No,No,No,No,No,No,No,No,Up,No
4,[40-50),No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No


In [227]:
order_map = {
    "age": age_order,
    "tolazamide": tolazamide_order,
}
ordinal_categories = [order_map.get(col, medical_order) for col in ordinal_cols]

In [228]:
ord_enc = OrdinalEncoder(categories=ordinal_categories)
df_encoded[ordinal_cols] = (ord_enc.fit_transform(df_encoded[ordinal_cols])).astype(int)

print("Порядковые столбцы после кодирования")

df_encoded[ordinal_cols].head()

Порядковые столбцы после кодирования


,age,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,miglitol,tolazamide,insulin,glyburide-metformin
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,3,0
2,2,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,0,0,0,0,3,0
4,4,0,0,0,0,0,1,0,0,0,0,0,0,1,0


#### One Hot Encoding

In [229]:
print("Размер датафрейма до кодирования")
df_encoded.shape

Размер датафрейма до кодирования


(101763, 43)

In [230]:
nominal_cols = [
    "race", "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3",
    "max_glu_serum", "A1Cresult", "change", "diabetesMed",
]

In [231]:
df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=True, dtype=int)

print("Размер датафрейма после кодирования")
df_encoded.shape

Размер датафрейма после кодирования


(101763, 162)

In [232]:
df_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 101763 entries, 0 to 101762
Columns: 162 entries, encounter_id to diabetesMed_Yes
dtypes: int64(162)
memory usage: 125.8 MB


In [233]:
df_encoded.to_csv("../data/processed/diabetic_data_encoded.csv")